# **4.0a**

## **Setup**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Master dataset = HIST + FORECAST + SCENARIO (delta)
# - HIST: engineered_ethnic_composition_full.csv
# - FORECAST: ethnic_forecasts_future_models.csv   (from 3.2)
# - SCENARIO: delta_mig_2025.csv (country-level Δmigrants) from 3.3
#   -> expanded to ethnic-group Δcount using baseline forecast shares in 2025
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

### Paths

In [3]:
BASE_DIR = Path("/content/drive/MyDrive/FYP")

ENG_DIR   = BASE_DIR / "data" / "engineered"
MODEL_DIR = BASE_DIR / "data" / "modeling"
OUT_DIR   = BASE_DIR / "data" / "master"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HIST_PATH      = ENG_DIR   / "engineered_ethnic_composition_full.csv"
FORECAST_PATH  = MODEL_DIR / "ethnic_forecasts_future_models.csv"
SCEN_PATH      = MODEL_DIR / "scenario_deltas_2025.csv"

OUT_MASTER = OUT_DIR / "ethnic_demography_master.csv"

### Settings

In [4]:
DEST_ISO3 = {"USA", "MYS", "IDN"}        # keep your scope
HIST_MAX_YEAR = 2024
SCENARIO_YEAR = 2025

# choose ONE baseline forecast model to store in master
BASELINE_FORECAST_MODEL = None  # e.g. "XGB_BASE" or keep None for auto-pick

NORMALIZE_SHARES = True
EPS = 1e-12

### Helpers

In [5]:
def _std_iso3(x):
    return x.astype(str).str.strip().str.upper()

def require_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}\nExisting: {df.columns.tolist()}")

def normalize_shares(df, keys=("iso3","year"), share_col="share"):
    out = df.copy()
    out[share_col] = pd.to_numeric(out[share_col], errors="coerce").clip(lower=0.0)
    s = out.groupby(list(keys))[share_col].transform("sum")
    out[share_col] = np.where(s > EPS, out[share_col] / s, out[share_col])
    return out

MASTER_COLS = [
    "iso3","year","ethnic_group",
    "share","count","total_pop",
    "model_name","view_type",
    "scenario_name",
    "delta_migrants","delta_in","delta_out",
    "delta_count",
    "base_share","shock_share","delta_share",
    "total_pop_shock"
]

def ensure_schema(df):
    out = df.copy()
    for c in MASTER_COLS:
        if c not in out.columns:
            out[c] = pd.NA
    return out[MASTER_COLS]

## **Load and build HIST**

In [6]:
hist = pd.read_csv(HIST_PATH)

if "iso3" not in hist.columns and "dest_iso3" in hist.columns:
    hist = hist.rename(columns={"dest_iso3": "iso3"})

require_cols(hist, ["iso3","year","ethnic_group"], "HIST")

# share
if "share_interp" in hist.columns:
    hist["share"] = hist["share_interp"]
elif "share" in hist.columns:
    hist["share"] = hist["share"]
else:
    raise ValueError("HIST needs share_interp or share")

# total_pop
if "total_pop_interp" in hist.columns:
    hist["total_pop"] = hist["total_pop_interp"]
elif "total_pop" in hist.columns:
    hist["total_pop"] = hist["total_pop"]
else:
    hist["total_pop"] = np.nan

hist["iso3"] = _std_iso3(hist["iso3"])
hist["year"] = pd.to_numeric(hist["year"], errors="coerce").astype(int)
hist["ethnic_group"] = hist["ethnic_group"].astype(str)

hist = hist[hist["iso3"].isin(DEST_ISO3)].copy()

hist["share"] = pd.to_numeric(hist["share"], errors="coerce")
hist["total_pop"] = pd.to_numeric(hist["total_pop"], errors="coerce")
hist["count"] = hist["share"] * hist["total_pop"]

hist_block = pd.DataFrame({
    "iso3": hist["iso3"],
    "year": hist["year"],
    "ethnic_group": hist["ethnic_group"],
    "share": hist["share"],
    "count": hist["count"],
    "total_pop": hist["total_pop"],
    "model_name": "HIST",
    "view_type": "HIST",
})

if NORMALIZE_SHARES:
    hist_block = normalize_shares(hist_block, keys=("iso3","year"), share_col="share")
    hist_block["count"] = hist_block["share"] * hist_block["total_pop"]

hist_block = ensure_schema(hist_block)

## **Load and build FORECAST**

In [7]:
fc = pd.read_csv(FORECAST_PATH)
require_cols(fc, ["iso3","year","ethnic_group","share","model_name"], "FORECAST")

fc["iso3"] = _std_iso3(fc["iso3"])
fc["year"] = pd.to_numeric(fc["year"], errors="coerce").astype(int)
fc["ethnic_group"] = fc["ethnic_group"].astype(str)
fc["share"] = pd.to_numeric(fc["share"], errors="coerce")

# total_pop
if "total_pop_interp" in fc.columns:
    fc["total_pop"] = pd.to_numeric(fc["total_pop_interp"], errors="coerce")
elif "total_pop" in fc.columns:
    fc["total_pop"] = pd.to_numeric(fc["total_pop"], errors="coerce")
else:
    raise ValueError("FORECAST needs total_pop_interp or total_pop to compute counts cleanly.")

fc = fc[fc["iso3"].isin(DEST_ISO3)].copy()

available_models = sorted(fc["model_name"].dropna().unique().tolist())
if not available_models:
    raise ValueError("No model_name values found in forecast file.")

if BASELINE_FORECAST_MODEL is None:
    preferred = [m for m in available_models if ("_BASE" in m or m.endswith("BASE"))]
    BASELINE_FORECAST_MODEL = preferred[0] if preferred else available_models[0]

fc = fc[fc["model_name"] == BASELINE_FORECAST_MODEL].copy()

# optional: keep only forecast years if a type column exists
if "type" in fc.columns:
    fc["type"] = fc["type"].astype(str).str.lower()
    fc = fc[fc["type"] != "historical"].copy()
else:
    fc = fc[fc["year"] > HIST_MAX_YEAR].copy()

fc["count"] = fc["share"] * fc["total_pop"]

forecast_block = pd.DataFrame({
    "iso3": fc["iso3"],
    "year": fc["year"],
    "ethnic_group": fc["ethnic_group"],
    "share": fc["share"],
    "count": fc["count"],
    "total_pop": fc["total_pop"],
    "model_name": fc["model_name"],
    "view_type": "FORECAST",
})

if NORMALIZE_SHARES:
    forecast_block = normalize_shares(forecast_block, keys=("iso3","year"), share_col="share")
    forecast_block["count"] = forecast_block["share"] * forecast_block["total_pop"]

forecast_block = ensure_schema(forecast_block)

## **Load Δmigrants**

In [8]:
scen = pd.read_csv(SCEN_PATH)

require_cols(
    scen,
    ["iso3","ethnic_group","year","scenario","base_share","shock_share",
     "delta_share","total_pop_interp","delta_count","delta_migrants_2025"],
    "SCENARIO"
)

# Optional columns
for c in ["delta_in","delta_out"]:
    if c not in scen.columns:
        scen[c] = 0.0

scen["iso3"] = _std_iso3(scen["iso3"])
scen["year"] = pd.to_numeric(scen["year"], errors="coerce").astype(int)
scen["ethnic_group"] = scen["ethnic_group"].astype(str)

scen = scen[scen["iso3"].isin(DEST_ISO3)].copy()
scen = scen[scen["year"] == SCENARIO_YEAR].copy()

# rename for master consistency
scen = scen.rename(columns={
    "scenario": "scenario_name",
    "total_pop_interp": "total_pop",
    "delta_migrants_2025": "delta_migrants",
})

# numeric
for c in ["base_share","shock_share","delta_share","delta_count","total_pop","delta_migrants","delta_in","delta_out"]:
    scen[c] = pd.to_numeric(scen[c], errors="coerce")

# country-level shock population (repeated per group)
scen["total_pop_shock"] = scen["total_pop"] + scen["delta_migrants"]

# IMPORTANT: for SCENARIO rows, expose SHOCKED series to the dashboard:
# share = shock_share
# count = shock_share * total_pop_shock
scen["share"] = scen["shock_share"]

if NORMALIZE_SHARES:
    # normalize shocked shares per iso3+scenario (safer for treemap/stacked area)
    scen = normalize_shares(scen, keys=("iso3","scenario_name"), share_col="share")

scen["count"] = scen["share"] * scen["total_pop_shock"]

# attach model_name from baseline forecast (so dashboard knows which forecast engine was used)
model_lookup = (forecast_block[forecast_block["year"] == SCENARIO_YEAR][["iso3","model_name"]]
                .drop_duplicates())
if model_lookup.empty:
    # fallback: use baseline model label anyway
    model_lookup = pd.DataFrame({"iso3": list(DEST_ISO3), "model_name": BASELINE_FORECAST_MODEL})

scen = scen.merge(model_lookup, on="iso3", how="left")

scen_block = pd.DataFrame({
    "iso3": scen["iso3"],
    "year": scen["year"],
    "ethnic_group": scen["ethnic_group"],
    "share": scen["share"],
    "count": scen["count"],
    "total_pop": scen["total_pop"],
    "model_name": scen["model_name"],
    "view_type": "SCENARIO",
    "scenario_name": scen["scenario_name"],
    "delta_migrants": scen["delta_migrants"],
    "delta_in": scen["delta_in"],
    "delta_out": scen["delta_out"],
    "delta_count": scen["delta_count"],
    "base_share": scen["base_share"],
    "shock_share": scen["shock_share"],
    "delta_share": scen["delta_share"],
    "total_pop_shock": scen["total_pop_shock"],
})

scen_block = ensure_schema(scen_block)

Expand scenario deltas to ethnic groups using baseline 2025 shares

## **Concatenate**

In [9]:
master = pd.concat([hist_block, forecast_block, scen_block], ignore_index=True)

# stable types
master["iso3"] = _std_iso3(master["iso3"])
master["year"] = pd.to_numeric(master["year"], errors="coerce").astype("Int64")
master["share"] = pd.to_numeric(master["share"], errors="coerce")
master["count"] = pd.to_numeric(master["count"], errors="coerce")
master["total_pop"] = pd.to_numeric(master["total_pop"], errors="coerce")
master["total_pop_shock"] = pd.to_numeric(master["total_pop_shock"], errors="coerce")

master = master.sort_values(["iso3","year","view_type","scenario_name","ethnic_group"]).reset_index(drop=True)



/tmp/ipython-input-3155837519.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master = pd.concat([hist_block, forecast_block, scen_block], ignore_index=True)


In [10]:
master.to_csv(OUT_MASTER, index=False)
print("Saved:", OUT_MASTER, "| shape:", master.shape)

Saved: /content/drive/MyDrive/FYP/data/master/ethnic_demography_master.csv | shape: (822, 17)


In [19]:
master

,iso3,year,ethnic_group,share,count,total_pop,model_name,view_type,scenario_name,delta_migrants,delta_in,delta_out,delta_count,base_share,shock_share,delta_share,total_pop_shock
0,IDN,2000,"Banjar, Melayu Banjar",0.017386,3.496273e+06,2.010922e+08,HIST,HIST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IDN,2000,Banten,0.020454,4.113162e+06,2.010922e+08,HIST,HIST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IDN,2000,Betawi,0.025072,5.041688e+06,2.010922e+08,HIST,HIST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IDN,2000,"Bugis, Ugi",0.024916,5.010423e+06,2.010922e+08,HIST,HIST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IDN,2000,Jawa,0.416490,8.375285e+07,2.010922e+08,HIST,HIST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,USA,2035,Black or African American,0.109745,4.075452e+07,3.713570e+08,ET_BASE,FORECAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
818,USA,2035,Native Hawaiian and other Pacific Islander,0.077876,2.891977e+07,3.713570e+08,ET_BASE,FORECAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
819,USA,2035,Some other race,0.094166,3.496909e+07,3.713570e+08,ET_BASE,FORECAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
820,USA,2035,Two or more races,0.101408,3.765860e+07,3.713570e+08,ET_BASE,FORECAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Validation

In [11]:
# A) HIST/FORECAST: sum(count) should be close to total_pop (after normalization)
for vt in ["HIST","FORECAST"]:
    tmp = master[master["view_type"] == vt].copy()
    chk = (tmp.groupby(["iso3","year"])
           .agg(sum_count=("count","sum"), pop=("total_pop","first"), sum_share=("share","sum"))
           .reset_index())
    chk["count_minus_pop"] = chk["sum_count"] - chk["pop"]
    print(f"\n[{vt}] share sum range:", chk["sum_share"].min(), "→", chk["sum_share"].max())
    print(f"[{vt}] (sum_count - pop) range:", chk["count_minus_pop"].min(), "→", chk["count_minus_pop"].max())



[HIST] share sum range: 0.9999999999999999 → 1.0
[HIST] (sum_count - pop) range: -2.9802322387695312e-08 → 3.725290298461914e-09

[FORECAST] share sum range: 0.9999999999999999 → 1.0000000000000002
[FORECAST] (sum_count - pop) range: 0.0 → 5.960464477539063e-08


In [12]:
# B) SCENARIO: sum(count) should be close to total_pop_shock per iso3+scenario
tmp = master[master["view_type"] == "SCENARIO"].copy()
if len(tmp):
    chk = (tmp.groupby(["iso3","scenario_name"])
           .agg(sum_count=("count","sum"),
                pop_shock=("total_pop_shock","first"),
                sum_share=("share","sum"),
                delta_migrants=("delta_migrants","first"),
                sum_delta_count=("delta_count","sum"))
           .reset_index())
    chk["count_minus_pop_shock"] = chk["sum_count"] - chk["pop_shock"]
    chk["delta_count_minus_delta_migrants"] = chk["sum_delta_count"] - chk["delta_migrants"]
    print("\n[SCENARIO] share sum range:", chk["sum_share"].min(), "→", chk["sum_share"].max())
    print("[SCENARIO] (sum_count - pop_shock) range:", chk["count_minus_pop_shock"].min(), "→", chk["count_minus_pop_shock"].max())
    print("[SCENARIO] (sum_delta_count - delta_migrants) range:", chk["delta_count_minus_delta_migrants"].min(), "→", chk["delta_count_minus_delta_migrants"].max())
else:
    print("\n[SCENARIO] No scenario rows found.")


[SCENARIO] share sum range: 0.9999999999999998 → 1.0000000000000002
[SCENARIO] (sum_count - pop_shock) range: -5.960464477539063e-08 → 5.960464477539063e-08
[SCENARIO] (sum_delta_count - delta_migrants) range: -4.656612873077393e-10 → 9.313225746154785e-10


# **4.0b**

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np

# ---------------- Paths ----------------
BASE_DIR = Path("/content/drive/MyDrive/FYP")

ENG_DIR   = BASE_DIR / "data" / "engineered"
MODEL_DIR = BASE_DIR / "data" / "modeling"
OUT_DIR   = BASE_DIR / "data" / "master"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Adjust if your OD hist file lives elsewhere
OD_HIST_PATH     = ENG_DIR   / "ims_od_inflow_outflow_engineered.csv"
OD_FORECAST_PATH = MODEL_DIR / "od_forecasts_future.csv"
OD_SCEN_PATH     = MODEL_DIR / "od_scenario_deltas_2025.csv"

OUT_OD_MASTER = OUT_DIR / "od_migration_master.csv"

# ---------------- Settings ----------------
DEST_ISO3 = {"USA", "MYS", "IDN"}        # keep your scope

# ---------------- Helpers ----------------
def _std_iso3_series(s):
    return s.astype(str).str.strip().str.upper()

def require_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}\nExisting: {df.columns.tolist()}")

OD_MASTER_COLS = [
    "iso3_orig","iso3_dest","year",
    "migrant_stock",          # HIST observed
    "migrant_stock_pred",     # baseline prediction (future)
    "migrant_stock_shock",    # scenario shocked value (2025 scenarios)
    "delta_in_od","delta_out_od","delta_stock_od",
    "model_name","view_type","scenario_name"
]

def ensure_od_schema(df):
    out = df.copy()
    for c in OD_MASTER_COLS:
        if c not in out.columns:
            out[c] = pd.NA
    return out[OD_MASTER_COLS]

In [15]:
od_hist = pd.read_csv(OD_HIST_PATH)
require_cols(od_hist, ["iso3_orig","iso3_dest","year","migrant_stock"], "OD_HIST")

od_hist["iso3_orig"] = _std_iso3_series(od_hist["iso3_orig"])
od_hist["iso3_dest"] = _std_iso3_series(od_hist["iso3_dest"])
od_hist["year"] = pd.to_numeric(od_hist["year"], errors="coerce").astype(int)
od_hist["migrant_stock"] = pd.to_numeric(od_hist["migrant_stock"], errors="coerce")

# scope to your countries (DEST)
od_hist = od_hist[od_hist["iso3_dest"].isin(DEST_ISO3)].copy()

od_hist_block = pd.DataFrame({
    "iso3_orig": od_hist["iso3_orig"],
    "iso3_dest": od_hist["iso3_dest"],
    "year": od_hist["year"],
    "migrant_stock": od_hist["migrant_stock"],
    "model_name": "HIST",
    "view_type": "HIST",
    "scenario_name": pd.NA
})
od_hist_block = ensure_od_schema(od_hist_block)

In [16]:


# ============================================================
# (2) FORECAST OD (baseline)
# ============================================================
od_fc = pd.read_csv(OD_FORECAST_PATH)
require_cols(od_fc, ["iso3_orig","iso3_dest","year","migrant_stock_pred","model_name"], "OD_FORECAST")

od_fc["iso3_orig"] = _std_iso3_series(od_fc["iso3_orig"])
od_fc["iso3_dest"] = _std_iso3_series(od_fc["iso3_dest"])
od_fc["year"] = pd.to_numeric(od_fc["year"], errors="coerce").astype(int)
od_fc["migrant_stock_pred"] = pd.to_numeric(od_fc["migrant_stock_pred"], errors="coerce")

od_fc = od_fc[od_fc["iso3_dest"].isin(DEST_ISO3)].copy()

od_forecast_block = pd.DataFrame({
    "iso3_orig": od_fc["iso3_orig"],
    "iso3_dest": od_fc["iso3_dest"],
    "year": od_fc["year"],
    "migrant_stock_pred": od_fc["migrant_stock_pred"],
    "model_name": od_fc["model_name"],
    "view_type": "FORECAST",
    "scenario_name": pd.NA
})
od_forecast_block = ensure_od_schema(od_forecast_block)

# ============================================================
# (3) SCENARIO OD (2025 scenarios)
# ============================================================
od_sc = pd.read_csv(OD_SCEN_PATH)
require_cols(
    od_sc,
    ["iso3_orig","iso3_dest","year","model_name","scenario",
     "migrant_stock_pred","delta_in_od","delta_out_od","delta_stock_od","migrant_stock_shock"],
    "OD_SCENARIO"
)

od_sc["iso3_orig"] = _std_iso3_series(od_sc["iso3_orig"])
od_sc["iso3_dest"] = _std_iso3_series(od_sc["iso3_dest"])
od_sc["year"] = pd.to_numeric(od_sc["year"], errors="coerce").astype(int)

for c in ["migrant_stock_pred","delta_in_od","delta_out_od","delta_stock_od","migrant_stock_shock"]:
    od_sc[c] = pd.to_numeric(od_sc[c], errors="coerce")

od_sc = od_sc[od_sc["iso3_dest"].isin(DEST_ISO3)].copy()

od_scen_block = pd.DataFrame({
    "iso3_orig": od_sc["iso3_orig"],
    "iso3_dest": od_sc["iso3_dest"],
    "year": od_sc["year"],
    "migrant_stock_pred": od_sc["migrant_stock_pred"],
    "migrant_stock_shock": od_sc["migrant_stock_shock"],
    "delta_in_od": od_sc["delta_in_od"],
    "delta_out_od": od_sc["delta_out_od"],
    "delta_stock_od": od_sc["delta_stock_od"],
    "model_name": od_sc["model_name"],
    "view_type": "SCENARIO",
    "scenario_name": od_sc["scenario"],
})
od_scen_block = ensure_od_schema(od_scen_block)

# ============================================================
# (4) CONCAT + SAVE
# ============================================================
od_master = pd.concat([od_hist_block, od_forecast_block, od_scen_block], ignore_index=True)

od_master = od_master.sort_values(
    ["iso3_dest","iso3_orig","year","view_type","scenario_name"],
    na_position="last"
).reset_index(drop=True)

od_master.to_csv(OUT_OD_MASTER, index=False)
print("Saved:", OUT_OD_MASTER, "| shape:", od_master.shape)

# Quick checks
print("\nOD master view_type counts:")
print(od_master["view_type"].value_counts(dropna=False))

print("\nScenario years present:")
print(sorted(od_master.loc[od_master["view_type"]=="SCENARIO","year"].dropna().unique().tolist())[:10])


/tmp/ipython-input-2550234061.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  od_master = pd.concat([od_hist_block, od_forecast_block, od_scen_block], ignore_index=True)


Saved: /content/drive/MyDrive/FYP/data/master/od_migration_master.csv | shape: (3850, 12)

OD master view_type counts:
view_type
SCENARIO    2860
HIST         660
FORECAST     330
Name: count, dtype: int64

Scenario years present:
[2025]


In [20]:
od_master

,iso3_orig,iso3_dest,year,migrant_stock,migrant_stock_pred,migrant_stock_shock,delta_in_od,delta_out_od,delta_stock_od,model_name,view_type,scenario_name
0,AFG,IDN,2000,78.0,NaN,NaN,NaN,NaN,NaN,HIST,HIST,NaN
1,AFG,IDN,2005,36.0,NaN,NaN,NaN,NaN,NaN,HIST,HIST,NaN
2,AFG,IDN,2010,1548.0,NaN,NaN,NaN,NaN,NaN,HIST,HIST,NaN
3,AFG,IDN,2015,6263.0,NaN,NaN,NaN,NaN,NaN,HIST,HIST,NaN
4,AFG,IDN,2020,7646.0,NaN,NaN,NaN,NaN,NaN,HIST,HIST,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3845,VNM,USA,2025,NaN,1.431834e+06,1.431834e+06,0.000000,0.0,0.000000,XGB_OD,SCENARIO,Conflict_internal_LOW
3846,VNM,USA,2025,NaN,1.431834e+06,1.517308e+06,85474.299543,0.0,85474.299543,XGB_OD,SCENARIO,Conflict_internal_LOW__Conflict_external_HIGH
3847,VNM,USA,2025,NaN,1.431834e+06,1.460325e+06,28491.433181,0.0,28491.433181,XGB_OD,SCENARIO,Conflict_internal_LOW__Conflict_external_LOW
3848,VNM,USA,2030,NaN,1.431834e+06,NaN,NaN,NaN,NaN,XGB_OD,FORECAST,NaN


# **OD Impact**

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# ============================================================
# 4.1 OD → Ethnic Allocation
# ============================================================

import pandas as pd
import numpy as np

# ----------------------------
# Load inputs
# ----------------------------
BASE_DIR = Path("/content/drive/MyDrive/FYP")

ENG_DIR   = BASE_DIR / "data" / "engineered"
MODEL_DIR = BASE_DIR / "data" / "modeling"
OUT_DIR   = BASE_DIR / "data" / "master"

od_master = pd.read_csv(OUT_OD_MASTER)
eth_map   = pd.read_csv( BASE_DIR / "data/panel/ethnic_mapping_fix.csv")
od_hist   = pd.read_csv(OD_HIST_PATH)

# Standardise keys
for df in [od_master, eth_map, od_hist]:
    for c in ["iso3_dest", "iso3_orig"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.upper().str.strip()

# ----------------------------
# Filter scenario rows only
# ----------------------------
od_scen = od_master[od_master["view_type"] == "SCENARIO"].copy()

# We only allocate inflow impacts
od_scen = od_scen[od_scen["delta_stock_od"] != 0].copy()

# ----------------------------
# Attach ethnic routing
# ----------------------------
od_eth = od_scen.merge(
    eth_map,
    on=["iso3_dest", "iso3_orig"],
    how="left"
)

# Fallback bucket
od_eth["dest_ethnic_bucket"] = od_eth["dest_ethnic_bucket"].fillna("Other")
od_eth["origin_ethnic_group"] = od_eth["origin_ethnic_group"].fillna("Unknown origin")

# ----------------------------
# Attach historical OD weights
# ----------------------------
od_weights = (
    od_hist
    .groupby(["iso3_dest", "iso3_orig"], as_index=False)
    .agg(weight=("origin_share_in_dest_migrants", "mean"))
)

od_eth = od_eth.merge(
    od_weights,
    on=["iso3_dest", "iso3_orig"],
    how="left"
)

# If no weight available, assume uniform
od_eth["weight"] = od_eth["weight"].fillna(1.0)

# ----------------------------
# Normalise weights per OD pair
# ----------------------------
od_eth["weight_norm"] = (
    od_eth["weight"] /
    od_eth.groupby(
        ["iso3_dest", "iso3_orig", "year", "scenario_name"]
    )["weight"].transform("sum")
)

# ----------------------------
# Allocate OD delta into ethnic buckets
# ----------------------------
od_eth["delta_migrants_ethnic"] = (
    od_eth["delta_stock_od"] * od_eth["weight_norm"]
)

# ----------------------------
# Aggregate to destination ethnic impact
# ----------------------------
od_eth_agg = (
    od_eth
    .groupby(
        [
            "iso3_dest",
            "year",
            "scenario_name",
            "dest_ethnic_bucket",
            "origin_ethnic_group"
        ],
        as_index=False
    )["delta_migrants_ethnic"]
    .sum()
)

# ----------------------------
# Save output
# ----------------------------
OD_ETHNIC_IMPACT_PATH = OUT_DIR / "od_ethnic_impact_2025.csv"
od_eth_agg.to_csv(
    OD_ETHNIC_IMPACT_PATH,
    index=False
)

print("OD ethnic allocation complete.")
display(od_eth_agg.head())

OD ethnic allocation complete.


,iso3_dest,year,scenario_name,dest_ethnic_bucket,origin_ethnic_group,delta_migrants_ethnic
0,IDN,2025,Aid_HIGH,"Banjar, Melayu Banjar",Malay,0.967181
1,IDN,2025,Aid_HIGH,Jawa,Javanese,0.967181
2,IDN,2025,Aid_HIGH,Jawa,Jawa Mapun,0.579414
3,IDN,2025,Aid_HIGH,Madura,Madura,0.967181
4,IDN,2025,Aid_HIGH,Other,Abelling,0.579414
